# Hybrid RRF Search Playground

Run hybrid RRF search directly from the module file, then save results to a log file.

In [23]:
import os, sys
from importlib.util import module_from_spec, spec_from_file_location

# Find repo root by locating the 'backend' directory
cwd = os.getcwd()
repo_root = cwd
while repo_root and not os.path.isdir(os.path.join(repo_root, 'backend')):
    parent = os.path.dirname(repo_root)
    if parent == repo_root:
        break
    repo_root = parent
print('repo_root=', repo_root)

module_path = os.path.join(repo_root, 'backend', 'app', 'agents', 'retrieval', 'cli_search_similar_chunks_direct_sql.py')
spec = spec_from_file_location('cli_search_similar_chunks_direct_sql', module_path)
module = module_from_spec(spec)
spec.loader.exec_module(module)
hybrid_rrf_search = module.hybrid_rrf_search

repo_root= c:\SKN_19\final_project\LLM


In [24]:
from dotenv import load_dotenv
load_dotenv(os.path.join(repo_root, 'backend', '.env'))

True

In [25]:
query = '민법 제110조'
results, sql_ms = hybrid_rrf_search(
    query,
    filter_document_type=['법률', '시행령'],
    result_limit=5,
)
print(f'sql_ms={sql_ms:.1f}')
for i, r in enumerate(results, 1):
    print(f"[{i}] rrf={r['rrf_score']:.4f} bm25={r['bm25_score']:.4f} vec={r['vector_similarity']:.4f} id={r['chunk_id']}")
    print(r['text'])
    print('-' * 60)

sql_ms=982.5
[1] rrf=0.0164 bm25=0.0000 vec=0.5270 id=제조물책임법_제8조
제8조(「민법」의 적용)
제조물의 결함으로 인한 손해배상책임에 관하여 이 법에 규정된 것을 제외하고는 「민법」에 따른다.
------------------------------------------------------------
[2] rrf=0.0161 bm25=0.0000 vec=0.4871 id=민법_제1118조
제1118조(준용규정)
제1001조, 제1008조, 제1010조의 규정은 유류분에 이를 준용한다.
------------------------------------------------------------
[3] rrf=0.0159 bm25=0.0000 vec=0.4723 id=민법_제105조
제105조(임의규정)
법률행위의 당사자가 법령 중의 선량한 풍속 기타 사회질서에 관계없는 규정과 다른 의사를 표시한 때에는 그 의사에 의한다.
------------------------------------------------------------
[4] rrf=0.0156 bm25=0.0000 vec=0.4636 id=상법_제726조의7
제726조의7(준용규정)
보증보험계약에 관하여는 그 성질에 반하지 아니하는 범위에서 보증채무에 관한 「민법」의 규정을 준용한다.
------------------------------------------------------------
[5] rrf=0.0154 bm25=0.0000 vec=0.4630 id=약관의 규제에 관한 법률 시행령_제5조의2
제5조의2()
------------------------------------------------------------


In [26]:
import json
from datetime import datetime

log_dir = os.path.join(repo_root, 'backend', 'logs', 'law_rag')
os.makedirs(log_dir, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = os.path.join(log_dir, f'law_rag_query_{timestamp}.json')
payload = {
    'timestamp': timestamp,
    'query': query,
    'sql_ms': sql_ms,
    'results': results,
}
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print('saved:', log_path)

saved: c:\SKN_19\final_project\LLM\backend\logs\law_rag\law_rag_query_20260127_183208.json
